In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

In [3]:
bookings = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\bookings.csv")

delays = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\delays.csv")

employees = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\employees.csv")

maintenance = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\maintenance.csv")

passengers = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\passengers.csv")

payments = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\payments.csv")

routes = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\routes.csv")

station = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\station.csv")

summary = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\summary.csv")

tickets = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\tickets.csv")

train_schedule = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\train_schedule.csv")

trains = pd.read_csv(r"C:\Users\katka\Railway_-Management-_-Analytics-\data\raw\trains.csv")

In [4]:
print(passengers.head())

print(bookings.head())

print(trains.head())


  passenger_id  passenger_name  age  gender                   email  \
0    PAS000001    Sunita Gupta   64    Male    passenger1@yahoo.com   
1    PAS000002     Kiran Patel   66  Female  passenger2@hotmail.com   
2    PAS000003       Pooja Rao   69  Female    passenger3@gmail.com   
3    PAS000004       Arun Nair   57  Female    passenger4@yahoo.com   
4    PAS000005  Arun Mukherjee   33    Male  passenger5@hotmail.com   

            phone    city loyalty_tier registration_date  total_trips  
0  +91 9837310642  Mumbai          NaN        2024-10-17          102  
1  +91 9581527840    Pune       Silver        2018-12-12            8  
2  +91 8756666624   Delhi       Silver        2019-09-12           83  
3  +91 8898167446    Pune          NaN        2021-10-11           14  
4  +91 9732742483   Delhi          NaN        2024-11-24          105  
  booking_id  ticket_id booking_date booking_channel  base_fare  \
0  BK0000001  TK0000001   2024-04-23         Counter   11190.27   
1  BK00

In [5]:
datasets = [
    passengers,
    trains,
    station,
    routes,
    bookings,
    tickets,
    payments,
    employees,
    train_schedule,
    delays,
    maintenance
]

for df in datasets:
    df.columns = (
        df.columns
        .str.strip()
        .str.lower()
        .str.replace(" ", "_")
    )

print("Columns cleaned successfully.")

Columns cleaned successfully.


In [6]:
bookings["total_amount"] = bookings["total_amount"].fillna(0)

tickets["fare"] = tickets["fare"].fillna(
    tickets["fare"].mean()
)

passengers["city"] = passengers["city"].fillna(
    "Unknown"
)

In [7]:
passengers.drop_duplicates(inplace=True)
bookings.drop_duplicates(inplace=True)
tickets.drop_duplicates(inplace=True)
payments.drop_duplicates(inplace=True)

In [8]:
bookings["booking_date"] = pd.to_datetime(bookings["booking_date"])

payments["payment_date"] = pd.to_datetime(payments["payment_date"])

maintenance["start_date"] = pd.to_datetime(
    maintenance["start_date"]
)

maintenance["end_date"] = pd.to_datetime(
    maintenance["end_date"]
)

In [9]:
bookings["booking_month"] = (
    bookings["booking_date"].dt.month
)

bookings["booking_year"] = (
    bookings["booking_date"].dt.year
)

bookings["day_name"] = (
    bookings["booking_date"].dt.day_name()
)

bookings["booking_type"] = np.where(
    bookings["day_name"].isin(
        ["Saturday", "Sunday"]
    ),
    "Weekend",
    "Weekday"
)

In [10]:
passenger_ticket = pd.merge(
    passengers,
    tickets,
    on="passenger_id",
    how="inner"
)

print(passenger_ticket.shape)

(8000, 20)


In [11]:
full_data = pd.merge(
    passenger_ticket,
    bookings,
    on="ticket_id",
    how="inner"
)

print(full_data.shape)

(8000, 32)


In [12]:
full_data = pd.merge(
    full_data,
    train_schedule,
    on="schedule_id",
    how="inner"
)

print(full_data.shape)

(8000, 40)


In [13]:
full_data = pd.merge(
    full_data,
    trains,
    on="train_id",
    how="inner"
)

print(full_data.shape)

(8000, 48)


In [14]:
full_data = pd.merge(
    full_data,
    payments,
    on="booking_id",
    how="left"
)

print(full_data.shape)

(8000, 56)


In [15]:
full_data.to_csv(
    "../data/cleaned/railway_management_system_final.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.


In [17]:
monthly_revenue = (
    bookings.groupby("booking_month")["total_amount"]
    .sum()
    .reset_index()
)

print(monthly_revenue.head())

   booking_month  total_amount
0              1    1911258.98
1              2    1815261.90
2              3    1962373.79
3              4    1756090.58
4              5    1806337.79


In [18]:
payment_analysis = (
    payments.groupby("payment_method")
    .size()
    .reset_index(name="total_transactions")
)

payment_analysis["percentage_share"] = (
    payment_analysis["total_transactions"]
    / payment_analysis["total_transactions"].sum()
) * 100

print(payment_analysis.head())

  payment_method  total_transactions  percentage_share
0           Cash                1141           14.2625
1    Credit Card                1180           14.7500
2     Debit Card                1146           14.3250
3            EMI                1152           14.4000
4    Net Banking                1188           14.8500


In [19]:
revenue_by_train = (
    full_data.groupby("train_name")["total_amount"]
    .sum()
    .sort_values(ascending=False)
    .reset_index()
)

print(revenue_by_train.head())

             train_name  total_amount
0       Kolkata Mail 14     299259.88
1  Bengaluru Express 71     295323.77
2    Sampark Kranti 125     280257.53
3   Shatabdi Express 29     256966.75
4       Chennai Mail 84     256118.50


In [20]:
import os

os.makedirs("../outputs", exist_ok=True)

In [21]:
monthly_revenue.to_csv(
    "../outputs/monthly_revenue.csv",
    index=False
)

payment_analysis.to_csv(
    "../outputs/payment_method_analysis.csv",
    index=False
)

revenue_by_train.to_csv(
    "../outputs/revenue_by_train.csv",
    index=False
)

print("Export completed successfully.")

Export completed successfully.
